# Module 3: KPI Engineering & Feature Creation

### 🔹 STEP 1: Load Clean Dataset

- Loads structured dataset from Module 2.
- All numeric fields are already cleaned.
- This file will now be enriched with KPIs.

In [214]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv("military_cleaned.csv")

df.head()

,country,power_index_rank,power_index_score,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,Afghanistan,121,2.7342,40121552.0,15647405.0,8826741.0,842553.0,75000.0,0.0,90000.0,...,8.020000e+07,8.020000e+07,4.955400e+10,767000.0,503000.0,66000000.0,652230.0,0.0,5987.0,1200.0
1,Albania,77,1.7258,3107100.0,1522479.0,1292554.0,62142.0,7500.0,0.0,500.0,...,5.062300e+07,5.062300e+07,5.692000e+09,473000.0,255000.0,522000000.0,28748.0,362.0,691.0,41.0
2,Algeria,27,0.4849,47022473.0,22570787.0,19185169.0,752360.0,130000.0,150000.0,150000.0,...,1.007260e+11,4.796300e+10,4.504000e+12,0.0,3000.0,233000000.0,2381740.0,998.0,6734.0,0.0
3,Angola,59,1.1045,37202061.0,7440412.0,3720206.0,372021.0,107000.0,0.0,10000.0,...,5.514000e+09,1.397000e+09,3.430020e+11,0.0,0.0,0.0,1246700.0,1600.0,5369.0,1300.0
4,Argentina,32,0.5983,46994384.0,20677529.0,17575900.0,704916.0,108000.0,12370.0,20000.0,...,4.328000e+10,4.622800e+10,3.964640e+11,869000.0,2534000.0,799999000.0,2780400.0,4989.0,11968.0,11000.0


### 🔹 STEP 2: Define Required KPIs

We will compute:

1. Assets per Capita
2. Budget-to-GDP Ratio
3. Power Index Rank Gap

Using your actual column names:

total_military_aircraft

tanks

total_naval_fleet

total_military_manpower

defense_budget_usd

purchasing_power_parity_usd (used as GDP proxy)

power_index_rank

### 🔹 STEP 3: KPI Calculations

#### 1️⃣ Assets per Capita

* Combines major assets into one metric.
* Divides by manpower.
* np.where() prevents division-by-zero error.

In [215]:
df["total_assets"] = (
    df["total_military_aircraft"] +
    df["tanks"] +
    df["total_naval_fleet"]
)

# Avoid division by zero
df["assets_per_capita"] = np.where(
    df["total_military_manpower"] > 0,
    df["total_assets"] / df["total_military_manpower"],
    0
)

#### 2️⃣ Budget-to-GDP Ratio
- Measures defense spending relative to economic strength.
- Higher value = stronger military investment.

In [216]:
df["budget_to_gdp_ratio"] = np.where(
    df["purchasing_power_parity_usd"] > 0,
    df["defense_budget_usd"] / df["purchasing_power_parity_usd"],
    0
)

#### 3️⃣ Power Index Rank Gap
- Top country has rank gap = 0.
- Shows how far each country is from strongest power.

In [217]:
top_rank = df["power_index_rank"].min()

df["power_index_rank_gap"] = df["power_index_rank"] - top_rank

### 🔹 STEP 4: Metadata Enrichment

We now add:
- region
- continent
- alliance_flag (NATO example)

In [218]:
pip install pycountry pycountry-convert

Note: you may need to restart the kernel to use updated packages.


In [219]:
import pycountry
import pycountry_convert as pc

def get_region(country_name):
    try:
        # Convert country name to ISO Alpha-2 code
        country_code = pycountry.countries.lookup(country_name).alpha_2
        
        # Convert ISO code to continent code
        continent_code = pc.country_alpha2_to_continent_code(country_code)
        
        # Convert continent code to continent name
        continent_name = pc.convert_continent_code_to_continent_name(continent_code)
        
        return continent_name
    
    except:
        return "Unknown"

df["region"] = df["country"].apply(get_region)

In [220]:
df["region"].value_counts()

region
Asia             43
Europe           37
Africa           36
South America    11
North America    10
Unknown           6
Oceania           2
Name: count, dtype: int64

NATO Flag Example

In [221]:
nato_countries = [
    "Albania", "Belgium", "Bulgaria", "Canada", "Croatia",
    "Czech Republic", "Denmark", "Estonia", "Finland", "France",
    "Germany", "Greece", "Hungary", "Iceland", "Italy",
    "Latvia", "Lithuania", "Luxembourg", "Montenegro",
    "Netherlands", "North Macedonia", "Norway", "Poland",
    "Portugal", "Romania", "Slovakia", "Slovenia",
    "Spain", "Sweden", "Turkey",
    "United Kingdom", "United States"
]

df["alliance_flag"] = df["country"].apply(
    lambda x: "NATO" if x in nato_countries else "Non-NATO"
)

### 🔹 STEP 5: Validate KPI Values

Check Division by Zero

In [222]:
df[df["total_military_manpower"] == 0]

,country,power_index_rank,power_index_score,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,...,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km,total_assets,assets_per_capita,budget_to_gdp_ratio,power_index_rank_gap,region,alliance_flag


#### Check KPI Ranges
Expected:
- Assets per capita → small decimal (<1 usually)
- Budget ratio → typically 0.01–0.10

In [223]:
df[["assets_per_capita", "budget_to_gdp_ratio"]].describe()

,assets_per_capita,budget_to_gdp_ratio
count,145.000000,145.000000
mean,0.000078,0.019467
std,0.000117,0.029967
min,0.000000,0.000085
25%,0.000017,0.005833
50%,0.000039,0.012138
75%,0.000088,0.023780
max,0.000705,0.308121


Manual Validation (Example)

In [224]:
df[df["country"] == "India"][
    ["total_assets", "total_military_manpower", "assets_per_capita"]
]

,total_assets,total_military_manpower,assets_per_capita
53,6439.0,662290299.0,0.00001


In [225]:
df[df["country"] == "India"]
df[df["country"] == "United States"]

,country,power_index_rank,power_index_score,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,...,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km,total_assets,assets_per_capita,budget_to_gdp_ratio,power_index_rank_gap,region,alliance_flag
137,United States,1,0.0741,341963408.0,150463900.0,124816644.0,4445524.0,1333030.0,799500.0,0.0,...,9833517.0,19924.0,12002.0,41009.0,18163.0,0.000121,0.032384,0,North America,NATO


In [226]:
final_columns = [
    "country",
    "power_index_rank",
    "total_military_manpower",
    "total_military_aircraft",
    "tanks",
    "total_naval_fleet",
    "defense_budget_usd",
    "purchasing_power_parity_usd",
    "assets_per_capita",
    "budget_to_gdp_ratio",
    "power_index_rank_gap",
    "region",
    "alliance_flag"
]

# 🔥 Overwrite df itself
df = df[final_columns]

df.head()

,country,power_index_rank,total_military_manpower,total_military_aircraft,tanks,total_naval_fleet,defense_budget_usd,purchasing_power_parity_usd,assets_per_capita,budget_to_gdp_ratio,power_index_rank_gap,region,alliance_flag
0,Afghanistan,121,15647405.0,5.0,0.0,0.0,1.450000e+08,8.223800e+10,3.195418e-07,0.001763,120,Asia,Non-NATO
1,Albania,77,1522479.0,20.0,0.0,33.0,7.200372e+08,5.136000e+10,3.481165e-05,0.014019,76,Europe,NATO
2,Algeria,27,22570787.0,620.0,1485.0,111.0,2.500000e+10,7.229120e+11,9.818001e-05,0.034582,26,Africa,Non-NATO
3,Angola,59,7440412.0,278.0,319.0,32.0,3.120000e+10,2.782390e+11,8.453833e-05,0.112134,58,Africa,Non-NATO
4,Argentina,32,20677529.0,241.0,362.0,42.0,9.939198e+08,1.213000e+12,3.119328e-05,0.000819,31,South America,Non-NATO


Round KPI Values

In [227]:
df_final["assets_per_capita"] = df_final["assets_per_capita"].round(6)
df_final["budget_to_gdp_ratio"] = df_final["budget_to_gdp_ratio"].round(4)

### 🔹 STEP 7: Export Final Dataset

In [228]:
df.to_csv("military_final.csv", index=False)

In [229]:
df.to_excel("military_final.xlsx", index=False)